> **Sentetik veri.** Bu defter `python -m sentetik.uret` ile üretilen uydurma veriyle çalışır. Sayılar gerçek bir şirkete ait değildir ve gerçek projenin sonuçlarını temsil etmez; amaç yöntemi uçtan uca göstermektir. *(Synthetic data: numbers are not real results.)*

# Sinyal — 04 · Kalibrasyon, açıklama (SHAP) ve sunum grafikleri

**Test zaten açıldı (03).** Bu defter test sonuçlarını değiştirmez; olasılıkları kalibre eder, modeli açıklar ve sunum grafiklerini üretir.

**Kalibrasyon yöntemi (sonuçlar görülmeden yazıldı):**
- **M0 ham:** LightGBM B skoru.
- **M1 izotonik:** Skor → olasılık eşlemesi, hedeften önce etiketi kapanmış *zaman dışı* skorlarla öğrenilir.
  - Test için: B modeli 2024-10 → 2025-04 kesimleriyle eğitilir, 2025-07 kesimini skorlar. İzotonik bu skorlarla öğrenilir. Bu, test modelinin yapısının aynısıdır: 3 ay önceye kadar eğit, sonraki kesimi skorla.
  - Test2 için: test modelinin test kesimi skorları kullanılır (etiketleri 2025-12-30'da kapanıyor; 2026-01-01'de biliniyor).
- **M2 düzey:** Logit kaydırma. Ortalama olasılık = geçen yılın aynı ayının etiket oranı (test ← 2024-10, test2 ← 2025-01). Bu oran skorlama anında bilinir.
- **Ana kalibrasyon = M1 + M2.** Sıralama ham skorla yapılır; olasılık değeri (aksiyon matrisi, kâr eğrisi) kalibre değerden alınır. İzotonik eşleme bağlı değerler üretebilir; sıralamayı bozmamak için bu ayrım yapılıyor.

**Maskeleme (KVKK):** Grafiklerde müşteri sayısı ve TL yok; yalnızca oranlar, puanlar ve endeksler.

In [ ]:
import sys, json
from pathlib import Path
KOK = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(KOK))

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.veri import tablo_yukle
from src.ozellikler import hazirla, model_b_ozellikleri, BAYRAK, DAVRANIS_SAYIM, DAVRANIS_DIGER
from src.degerlendirme import hedef_egitim_indeksleri, capture_at_k
from src.modeller import egit, skorla, matris
from src.kalibrasyon import izotonik_uydur, duzey_kaydir, guvenilirlik_tablosu, kalibrasyon_ozeti

SONUC = KOK / "docs" / "sonuclar"; FIG = KOK / "notebooks" / "figurler"; FIG.mkdir(exist_ok=True)
VERI = KOK / "data"
TOHUM = 42
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
pd.set_option("display.float_format", "{:.4f}".format)

st = pd.read_parquet(VERI / "03_skorlar_test.parquet")
st2 = pd.read_parquet(VERI / "03_skorlar_test2.parquet")
a = tablo_yukle("egitim_a"); b = tablo_yukle("egitim_b")
df = hazirla(a, b)   # v1.2 varsayılanları: yuzdelik_aktif, web_payi_goreli
df["y_etiket"] = df["y_etiket"].astype(int)
KOL_B = model_b_ozellikleri(df)
assert len(KOL_B) == 38
oran = df.groupby("kesim")["y_etiket"].mean()
HEDEF_ORAN = {"test": float(oran[pd.Timestamp("2024-10-01")]), "test2": float(oran[pd.Timestamp("2025-01-01")])}
HEDEF_ORAN

## 1. Kalibrasyon kaynağı (zaman dışı skorlar)

In [ ]:
# Test için: 2024-10 → 2025-04 ile eğit, 2025-07'yi skorla (test modeliyle aynı 3 aylık boşluk)
kaynak_kesim = pd.Timestamp("2025-07-01")
egt = hedef_egitim_indeksleri(df, kaynak_kesim, ("egitim",))
kaynak = df[df["kesim"] == kaynak_kesim]
model_k = egit("lightgbm", df.loc[egt], KOL_B)
skor_k = skorla("lightgbm", model_k, kaynak, KOL_B)
print("kaynak eğitim kesimleri:", sorted(df.loc[egt, "kesim"].dt.strftime("%Y-%m").unique()))

ISO = {"test": izotonik_uydur(skor_k, kaynak["y_etiket"]),
       "test2": izotonik_uydur(st["skor_lightgbm_B"], st["y_etiket"])}   # test etiketleri 2026-01-01'de biliniyor

## 2. Kalibrasyon sonuçları (öncesi / sonrası)

In [ ]:
KAL = {}; ozet = []
for ad, d in [("test", st), ("test2", st2)]:
    ham = d["skor_lightgbm_B"].to_numpy()
    m1 = ISO[ad].predict(ham)
    m12 = duzey_kaydir(m1, HEDEF_ORAN[ad])
    m2 = duzey_kaydir(ham, HEDEF_ORAN[ad])
    KAL[ad] = {"M0 ham": ham, "M1 izotonik": m1, "M2 düzey": m2, "M1+M2 (ana)": m12}
    for yontem, p in KAL[ad].items():
        ozet.append({"bolum": ad, "yontem": yontem, **kalibrasyon_ozeti(d["y_etiket"], p)})
OZ = pd.DataFrame(ozet)
OZ.to_csv(SONUC / "04_kalibrasyon.csv", index=False)
OZ

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)
for ax, (ad, d) in zip(axs, [("test", st), ("test2", st2)]):
    for yontem, renk in [("M0 ham", "#9aa0a6"), ("M1+M2 (ana)", "#1a73e8")]:
        t = guvenilirlik_tablosu(d["y_etiket"], KAL[ad][yontem])
        ax.plot(t["tahmin"], t["gercek"], "o-", color=renk, label=yontem)
    ax.plot([0, 0.7], [0, 0.7], "--", color="#555", lw=1, label="mükemmel")
    ax.set_title(f"{ad}: tahmin edilen ve gerçekleşen alım oranı")
    ax.set_xlabel("Tahmin (dilim ortalaması)"); ax.legend()
axs[0].set_ylabel("Gerçekleşen (dilim ortalaması)")
plt.tight_layout(); plt.savefig(FIG / "04_kalibrasyon.png", bbox_inches="tight"); plt.show()

## 3. Sunum grafikleri: kümülatif kazanç ve model karşılaştırma (test)

In [ ]:
def kazanc(y, s, adim=np.linspace(0.01, 1, 100)):
    y = np.asarray(y); sira = np.argsort(-np.asarray(s, dtype=float), kind="mergesort")
    kum = np.cumsum(y[sira]) / y.sum()
    return np.array([kum[max(1, int(round(k * len(y)))) - 1] for k in adim])

ADIM = np.linspace(0.01, 1, 100)
EGRI = {"Kural (yakınlık)": "skor_kural", "RFM": "skor_rfm", "Model A (sipariş)": "skor_lightgbm_A",
        "Model B (sipariş + davranış)": "skor_lightgbm_B"}
K = pd.DataFrame({ad: kazanc(st["y_etiket"], st[k], ADIM) for ad, k in EGRI.items()}, index=ADIM.round(2))
K.to_csv(SONUC / "04_kazanc_egrisi_test.csv")

fig, ax = plt.subplots(figsize=(6.5, 5))
for (ad, renk) in zip(EGRI, ["#9aa0a6", "#fbbc04", "#34a853", "#1a73e8"]):
    ax.plot(ADIM * 100, K[ad] * 100, color=renk, lw=2 if "B" in ad else 1.5, label=ad)
ax.plot([0, 100], [0, 100], "--", color="#555", lw=1, label="Rastgele")
ax.axvline(10, color="#d93025", lw=0.8, ls=":")
ax.set_xlabel("Hedeflenen müşteri payı (%, skora göre)"); ax.set_ylabel("Yakalanan alıcı payı (%)")
ax.set_title("Kümülatif kazanç — test (Ekim 2025)"); ax.legend(loc="lower right")
plt.tight_layout(); plt.savefig(FIG / "04_kazanc_egrisi.png", bbox_inches="tight"); plt.show()
K.loc[[0.05, 0.1, 0.2, 0.3]] * 100

In [ ]:
M = pd.read_csv(SONUC / "03_metrikler.csv").query("bolum == 'test' and alt_kume == 'tum'")
M = M[(M["model"].isin(["kural", "rfm", "bgnbd"])) | (M["model"] == "lightgbm")]
M["ad"] = np.where(M["set"] == "-", M["model"].map({"kural": "Kural", "rfm": "RFM", "bgnbd": "BG/NBD"}),
                   "LightGBM " + M["set"])
M = M.set_index("ad").loc[["Kural", "BG/NBD", "RFM", "LightGBM A", "LightGBM A+bayrak", "LightGBM B"]]
fig, axs = plt.subplots(1, 2, figsize=(11, 4))
for ax, (met, baslik) in zip(axs, [("capture@10", "İlk %10'da yakalanan alıcı payı"), ("pr_auc", "PR-AUC")]):
    renk = ["#9aa0a6"] * 3 + ["#34a853", "#81c995", "#1a73e8"]
    ax.barh(M.index, M[met], color=renk)
    for i, v in enumerate(M[met]):
        ax.text(v + 0.003, i, f"{v:.3f}", va="center", fontsize=9)
    ax.set_title(baslik); ax.invert_yaxis()
plt.tight_layout(); plt.savefig(FIG / "04_model_karsilastirma.png", bbox_inches="tight"); plt.show()

## 4. Deneyin sonucu: B − A alt gruplarda (%95 GA)

In [ ]:
AD = {"tum": "Tüm müşteriler", "karar": "Son siparişi > 30 gün", "kapsam": "Davranışı olan",
      "davranissiz": "Davranışı olmayan", "tek_siparis": "Tek siparişli", "tekrar": "Tekrar eden",
      "uzak_180": "Son siparişi > 180 gün"}
F = pd.read_csv(SONUC / "03_bootstrap.csv").query("model == 'lightgbm' and metrik == 'capture@10'")
fig, ax = plt.subplots(figsize=(8, 4.8))
for j, (bolum, renk, kay) in enumerate([("test", "#1a73e8", -0.15), ("test2", "#fbbc04", 0.15)]):
    f = F[F["bolum"] == bolum].set_index("alt_kume").loc[list(AD)]
    yy = np.arange(len(AD)) + kay
    ax.errorbar(f["fark"] * 100, yy, xerr=[(f["fark"] - f["alt_%2.5"]) * 100, (f["ust_%97.5"] - f["fark"]) * 100],
                fmt="o", color=renk, capsize=3, label=bolum)
ax.axvline(0, color="#555", lw=1); ax.axvline(1, color="#d93025", lw=0.8, ls=":", label="eşik: 1 puan")
ax.set_yticks(range(len(AD))); ax.set_yticklabels(AD.values()); ax.invert_yaxis()
ax.set_xlabel("B − A, ilk %10'da yakalanan alıcı payı farkı (yüzde puan)")
ax.set_title("Davranış verisinin katkısı alt gruplarda"); ax.legend()
plt.tight_layout(); plt.savefig(FIG / "04_alt_gruplar.png", bbox_inches="tight"); plt.show()

## 5. Açıklama: SHAP (LightGBM B, test modeli)

In [ ]:
ETIKET = {
    "siparis_12ay": "Sipariş sayısı (12 ay)", "siparis_30g": "Sipariş sayısı (30 gün)",
    "siparis_90g": "Sipariş sayısı (90 gün)", "siparis_180g": "Sipariş sayısı (180 gün)",
    "siparis_9ay": "Sipariş sayısı (9 ay)", "gun_son_siparis": "Son siparişten bu yana gün",
    "gun_ilk_siparis_12ay": "12 aydaki ilk siparişten bu yana gün", "ort_siparis_arasi_gun": "Siparişler arası ort. gün",
    "adet_12ay": "Ürün adedi (12 ay)", "ort_kalem": "Sipariş başına kalem", "pay_app": "Uygulamadan sipariş payı",
    "pay_mobil_web": "Mobil webden sipariş payı", "indirim_derinligi_9ay": "İndirim derinliği (9 ay)",
    "tutar_eksik_oran_9ay": "Tutarı eksik sipariş payı", "pay_tekstil": "Tekstil payı", "pay_aksesuar": "Aksesuar payı",
    "pay_ayakkabi": "Ayakkabı payı", "farkli_kategori2": "Farklı alt kategori sayısı", "uyelik_gun": "Üyelik süresi",
    "yas": "Yaş", "son_platform": "Son sipariş platformu", "ort_sepet_9ay_e": "Ortalama sepet (endeks)",
    "ciro_9ay_e": "Ciro, 9 ay (endeks)", "ort_birim_fiyat_9ay_e": "Ort. birim fiyat (endeks)",
    "davranis_var_30g": "Son 30 günde davranış var", "gun_son_aktivite": "Son ziyaretten bu yana gün",
    "web_gun_payi_y": "Web gün payı (sıra)", "aktif_gun_7g_y": "Aktif gün, 7 gün (sıra)",
    "aktif_gun_30g_y": "Aktif gün, 30 gün (sıra)", "urun_gor_7g_y": "Ürün görüntüleme, 7 gün (sıra)",
    "urun_gor_30g_y": "Ürün görüntüleme, 30 gün (sıra)", "liste_gor_30g_y": "Liste görüntüleme (sıra)",
    "arama_30g_y": "Arama (sıra)", "sepete_ekle_7g_y": "Sepete ekleme, 7 gün (sıra)",
    "sepete_ekle_30g_y": "Sepete ekleme, 30 gün (sıra)", "favori_30g_y": "Favori (sıra)",
    "odeme_baslat_30g_y": "Ödeme başlatma (sıra)", "toplam_event_30g_y": "Toplam etkileşim (sıra)",
}
DAVRANIS_KOL = set(BAYRAK + [f"{c}_y" for c in DAVRANIS_SAYIM] + ["gun_son_aktivite", "web_gun_payi_y"])

model_b = joblib.load(VERI / "modeller_03" / "test_lightgbm_B.joblib")
test_df = df[df["kesim"] == pd.Timestamp("2025-10-01")]
orn = test_df.sample(n=min(30000, len(test_df)), random_state=TOHUM)
X = matris(orn, KOL_B, lgbm=True)
katki = model_b.booster_.predict(X, pred_contrib=True)[:, :-1]   # LightGBM'in kendi SHAP değerleri (log-odds)
O = (pd.DataFrame({"ozellik": KOL_B, "ort_mutlak_shap": np.abs(katki).mean(axis=0)})
       .assign(ad=lambda t: t["ozellik"].map(ETIKET).fillna(t["ozellik"]),
               grup=lambda t: np.where(t["ozellik"].isin(DAVRANIS_KOL), "davranış", "sipariş/müşteri"))
       .sort_values("ort_mutlak_shap", ascending=False))
O.to_csv(SONUC / "04_shap_onem.csv", index=False)

ust = O.head(15).iloc[::-1]
fig, ax = plt.subplots(figsize=(7.5, 5.5))
ax.barh(ust["ad"], ust["ort_mutlak_shap"], color=np.where(ust["grup"] == "davranış", "#1a73e8", "#9aa0a6"))
ax.set_xlabel("Ortalama |SHAP| (log-odds)"); ax.set_title("En etkili 15 özellik (mavi: davranış)")
plt.tight_layout(); plt.savefig(FIG / "04_shap_onem.png", bbox_inches="tight"); plt.show()

# Davranışın toplam açıklamadaki payı: genel ve alt gruplarda
pay = []
for ad, maske in {"Tüm": np.ones(len(orn), bool), "Tek siparişli": (orn["siparis_12ay"] == 1).to_numpy(),
                  "Tekrar eden": (orn["siparis_12ay"] >= 2).to_numpy(),
                  "Davranışı olan": orn["davranis_var_30g"].astype(bool).to_numpy()}.items():
    m = np.abs(katki[maske]).mean(axis=0)
    pay.append({"grup": ad, "davranis_payi": m[[c in DAVRANIS_KOL for c in KOL_B]].sum() / m.sum()})
PAY = pd.DataFrame(pay); PAY.to_csv(SONUC / "04_shap_davranis_payi.csv", index=False)
PAY

In [ ]:
try:
    import shap
    ust15 = O.head(15)["ozellik"].tolist(); idx = [KOL_B.index(c) for c in ust15]
    Xg = X[ust15].copy()
    if "son_platform" in Xg:
        Xg["son_platform"] = Xg["son_platform"].cat.codes
    shap.summary_plot(katki[:, idx], Xg.rename(columns=ETIKET), show=False, max_display=15)
    plt.title("SHAP: özellik değeri yüksek (kırmızı) / düşük (mavi)")
    plt.savefig(FIG / "04_shap_beeswarm.png", bbox_inches="tight"); plt.show()
except Exception as hata:
    print("Beeswarm atlandı:", hata)

## 6. Kalibre skorları kaydet (05: aksiyon matrisi ve kâr eğrisi)

In [ ]:
for ad, d in [("test", st), ("test2", st2)]:
    k = d[["kesim", "musteri_id", "y_etiket", "y_siparis_90g", "y_ciro_90g", "gun_son_siparis",
           "siparis_12ay", "davranis_var_30g", "skor_lightgbm_A", "skor_lightgbm_B"]].copy()
    k["p_kalibre"] = KAL[ad]["M1+M2 (ana)"]
    k.to_parquet(VERI / f"04_skorlar_kalibre_{ad}.parquet", index=False)   # data/ gitignore'da
print("Kaydedildi: docs/sonuclar/04_*.csv ve notebooks/figurler/04_*.png")